In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Clean Dataset by Normalizing Tags

This step takes into account the potential duplicates identified previously (case variations, format variations, etc.) and consolidates them into a single, canonical tag entry. It normalizes tag keys and values, then aggregates their counts. For the canonical representation, it selects the original key/value pair with the highest frequency within each normalized group.

In [ ]:
import pandas as pd
import re

# Define normalize function (copied from previous cells to ensure it's available)
def normalize(s):
    """Normalize for comparison."""
    # Convert to string to handle non-string types (like floats for NaN) before lowercasing
    if pd.isna(s):
        return ""
    return re.sub(r'[ _:-]', ' ', str(s).lower()).strip()

# Load df_tags from CSV (if not already in memory, to resolve NameError)
# This assumes 'pakistan_unique_osm_tags.csv' has been created by previous steps (cell 'q9o5hqkT0ma9').
try:
    df_tags
except NameError:
    print("df_tags not found in memory, attempting to load from CSV.")
    try:
        # Corrected file path
        df_tags = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/datageospatial/pakistan_unique_osm_tags.csv')
        print(f"Loaded {len(df_tags):,} tags from /content/drive/MyDrive/ColabNotebooks/datageospatial/pakistan_unique_osm_tags.csv")
    except FileNotFoundError:
        print("Error: 'pakistan_unique_osm_tags.csv' not found at the specified path. Please ensure 'Step 1: Extract All Unique OSM Tags' has been fully executed and saved the CSV to the correct Google Drive location.")
        raise # Re-raise the error if the file isn't found


print("🔍 Cleaning dataset by normalizing tag keys and values...")

# Create normalized columns using the 'normalize' function already defined.
df_tags['normalized_key'] = df_tags['tag_key'].apply(normalize)
df_tags['normalized_value'] = df_tags['tag_value'].apply(normalize)

# Group by normalized key and value, and sum counts.
# For tag_key and tag_value, pick the original value that has the highest count within the group.
# This ensures a canonical representation based on popularity.
def get_canonical_tag_info(group):
    # Sort by count in descending order to pick the most frequent original tag
    sorted_group = group.sort_values(by='count', ascending=False)
    canonical_key = sorted_group.iloc[0]['tag_key']
    canonical_value = sorted_group.iloc[0]['tag_value']
    total_count = sorted_group['count'].sum()
    return pd.Series({
        'tag_key': canonical_key,
        'tag_value': canonical_value,
        'full_pair': f"{canonical_key}={canonical_value}" if canonical_value else canonical_key,
        'count': total_count
    })

df_tags_cleaned = df_tags.groupby(['normalized_key', 'normalized_value'], group_keys=False).apply(get_canonical_tag_info)
df_tags_cleaned = df_tags_cleaned.sort_values('count', ascending=False).reset_index(drop=True)

print(f"\n✅ Cleaned dataset has {len(df_tags_cleaned):,} unique tags (after normalization).")
print(f"   Original dataset had {len(df_tags):,} unique tags based on exact string match.")

print("\n📊 Top 20 Most Common Cleaned Tag Pairs:")
display(df_tags_cleaned.head(20))

In [ ]:
# Save the cleaned dataset to CSV
df_tags_cleaned.to_csv('pakistan_unique_osm_tags_cleaned.csv', index=False)
print(f"💾 Saved {len(df_tags_cleaned):,} cleaned unique tags to pakistan_unique_osm_tags_cleaned.csv")

# Download the cleaned file
from google.colab import files
files.download('pakistan_unique_osm_tags_cleaned.csv')

# OSM Pakistan Tag Analysis

This notebook extracts unique OSM tags from Pakistan's OpenStreetMap data and identifies potential duplicate tags.

## Setup
1. Upload `pakistan-latest.osm.pbf` to Colab (or mount Google Drive)
2. Run all cells

In [ ]:
# Install required packages
!pip install osmium pandas -q
print("Packages installed")

In [ ]:
# Option A: Upload file directly
from google.colab import files
import os

# Check if file already exists
if not os.path.exists('pakistan-latest.osm.pbf'):
    print("Upload pakistan-latest.osm.pbf:")
    uploaded = files.upload()
else:
    print(" File already exists")

In [ ]:
# Option B: Mount Google Drive (if file is in Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_FILE = '/content/drive/MyDrive/path/to/pakistan-latest.osm.pbf'

## Step 1: Extract All Unique OSM Tags

In [ ]:
import osmium
import pandas as pd
from collections import Counter, defaultdict
from tqdm import tqdm
import json

INPUT_FILE = "/content/drive/MyDrive/ColabNotebooks/datageospatial/pakistan-latest.osm.pbf"

class OSMTagExtractor(osmium.SimpleHandler):
    """Extract all unique OSM tags from the dataset."""

    def __init__(self):
        super().__init__()
        self.tag_keys = Counter()
        self.tag_values = defaultdict(Counter)
        self.tag_pairs = Counter()
        self.node_count = 0
        self.way_count = 0
        self.relation_count = 0
        self.elements_with_tags = 0

    def _process_tags(self, tags):
        if not tags:
            return
        self.elements_with_tags += 1
        for tag in tags:
            self.tag_keys[tag.k] += 1
            self.tag_values[tag.k][tag.v] += 1
            self.tag_pairs[f"{tag.k}={tag.v}"] += 1

    def node(self, n):
        self.node_count += 1
        self._process_tags(n.tags)

    def way(self, w):
        self.way_count += 1
        self._process_tags(w.tags)

    def relation(self, r):
        self.relation_count += 1
        self._process_tags(r.tags)

print("🔄 Extracting OSM tags (this takes ~2-3 minutes)...")
handler = OSMTagExtractor()
handler.apply_file(INPUT_FILE)

print(f"\n✅ Extraction Complete!")
print(f"   Nodes: {handler.node_count:,}")
print(f"   Ways: {handler.way_count:,}")
print(f"   Relations: {handler.relation_count:,}")
print(f"   Elements with tags: {handler.elements_with_tags:,}")
print(f"   Unique tag keys: {len(handler.tag_keys):,}")
print(f"   Unique key=value pairs: {len(handler.tag_pairs):,}")

In [ ]:
# Convert to DataFrame for easy analysis
rows = []
for pair, count in handler.tag_pairs.items():
    if "=" in pair:
        key, value = pair.split("=", 1)
    else:
        key, value = pair, ""
    rows.append({"tag_key": key, "tag_value": value, "full_pair": pair, "count": count})

df_tags = pd.DataFrame(rows)
df_tags = df_tags.sort_values('count', ascending=False).reset_index(drop=True)

print(f"\n📊 Top 20 Most Common Tag Pairs:")
df_tags.head(20)

In [ ]:
# Save to CSV
df_tags.to_csv('pakistan_unique_osm_tags.csv', index=False)
print(f"💾 Saved {len(df_tags):,} unique tags to pakistan_unique_osm_tags.csv")

## Step 2: Find Potential Duplicate Tags

This identifies tags written differently but meaning the same thing:
- **Case variations**: `Hospital` vs `hospital`
- **Format variations**: `place_of_worship` vs `place-of-worship`
- **Plural variations**: `shop` vs `shops`
- **Similar spelling**: `resturant` vs `restaurant`

In [ ]:
import re
from difflib import SequenceMatcher
from tqdm.notebook import tqdm

def normalize(s):
    """Normalize for comparison."""
    return re.sub(r'[_\-:]', ' ', s.lower()).strip()

def similarity(s1, s2):
    """Calculate similarity ratio."""
    return SequenceMatcher(None, s1, s2).ratio()

def is_case_variant(a, b):
    return a.lower() == b.lower() and a != b

def is_format_variant(a, b):
    return normalize(a) == normalize(b) and a != b

def is_plural(a, b):
    a, b = a.lower(), b.lower()
    return (a + 's' == b or b + 's' == a or
            a + 'es' == b or b + 'es' == a or
            (a.endswith('y') and a[:-1] + 'ies' == b) or
            (b.endswith('y') and b[:-1] + 'ies' == a))

SIMILARITY_THRESHOLD = 0.85
MIN_OCCURRENCES = 2

In [ ]:
# Find duplicate TAG KEYS
print("🔍 Analyzing tag keys for duplicates...")

# Aggregate counts per key
key_counts = df_tags.groupby('tag_key')['count'].sum().to_dict()
keys = [k for k, c in key_counts.items() if c >= MIN_OCCURRENCES]

key_duplicates = []

for i, key1 in enumerate(tqdm(keys, desc="Comparing keys")):
    for key2 in keys[i+1:]:
        if key1 == key2:
            continue

        reason = None
        sim = 0.0

        if is_case_variant(key1, key2):
            reason = "case_variation"
            sim = 1.0
        elif is_format_variant(key1, key2):
            reason = "format_variation"
            sim = 1.0
        elif is_plural(key1, key2):
            reason = "plural_variation"
            sim = 0.95
        elif len(key1) <= 20 and len(key2) <= 20:
            sim = similarity(key1.lower(), key2.lower())
            if sim >= SIMILARITY_THRESHOLD:
                reason = "similar_spelling"

        if reason:
            key_duplicates.append({
                "type": "key",
                "tag1": key1,
                "tag2": key2,
                "count1": key_counts[key1],
                "count2": key_counts[key2],
                "reason": reason,
                "similarity": round(sim, 3)
            })

print(f"\n✅ Found {len(key_duplicates)} potential duplicate TAG KEYS")

In [ ]:
# Display key duplicates
df_key_dups = pd.DataFrame(key_duplicates)
if not df_key_dups.empty:
    df_key_dups = df_key_dups.sort_values(['reason', 'similarity'], ascending=[True, False])
    print("\n📋 TAG KEY DUPLICATES:")
    display(df_key_dups)
else:
    print("No tag key duplicates found.")

In [ ]:
# Find duplicate TAG VALUES (for top 30 most used keys only)
print("🔍 Analyzing tag values for duplicates...")

top_keys = df_tags.groupby('tag_key')['count'].sum().nlargest(30).index.tolist()
value_duplicates = []

for key in tqdm(top_keys, desc="Checking values"):
    values = df_tags[df_tags['tag_key'] == key]
    values = values[values['count'] >= MIN_OCCURRENCES]
    val_list = values[['tag_value', 'count']].values.tolist()

    for i, (val1, c1) in enumerate(val_list):
        for val2, c2 in val_list[i+1:]:
            if val1 == val2 or len(val1) < 2 or len(val2) < 2:
                continue

            reason = None
            sim = 0.0

            if is_case_variant(val1, val2):
                reason = "case_variation"
                sim = 1.0
            elif is_format_variant(val1, val2):
                reason = "format_variation"
                sim = 1.0
            elif is_plural(val1, val2):
                reason = "plural_variation"
                sim = 0.95
            elif len(val1) >= 3 and len(val2) >= 3 and len(val1) <= 30 and len(val2) <= 30:
                sim = similarity(val1.lower(), val2.lower())
                if sim >= SIMILARITY_THRESHOLD:
                    reason = "similar_spelling"

            if reason:
                value_duplicates.append({
                    "type": "value",
                    "tag_key": key,
                    "tag1": f"{key}={val1}",
                    "tag2": f"{key}={val2}",
                    "count1": c1,
                    "count2": c2,
                    "reason": reason,
                    "similarity": round(sim, 3)
                })

print(f"\n✅ Found {len(value_duplicates)} potential duplicate TAG VALUES")

In [ ]:
# Display value duplicates grouped by reason
df_val_dups = pd.DataFrame(value_duplicates)
if not df_val_dups.empty:
    df_val_dups = df_val_dups.sort_values(['reason', 'similarity'], ascending=[True, False])

    for reason in df_val_dups['reason'].unique():
        print(f"\n📋 VALUE DUPLICATES - {reason.upper()}:")
        display(df_val_dups[df_val_dups['reason'] == reason].head(30))
else:
    print("No tag value duplicates found.")

In [ ]:
# Save all duplicates to CSV
all_dups = pd.concat([df_key_dups, df_val_dups], ignore_index=True) if not df_key_dups.empty or not df_val_dups.empty else pd.DataFrame()

if not all_dups.empty:
    all_dups.to_csv('potential_duplicate_tags.csv', index=False)
    print(f"💾 Saved {len(all_dups)} potential duplicates to potential_duplicate_tags.csv")

    # Download
    files.download('potential_duplicate_tags.csv')
else:
    print("No duplicates to save.")

In [ ]:
# Download the unique tags file too
files.download('pakistan_unique_osm_tags.csv')

In [ ]:
# Install dependencies
!pip install pandas -q
print("✅ Packages ready")

In [ ]:
# Upload the CSV file
from google.colab import files
import os

# Check if file exists
possible_names = [
    'pakistan_unique_osm_tags (2).csv',
    'pakistan_unique_osm_tags.csv'
]

INPUT_FILE = None
for name in possible_names:
    if os.path.exists(name):
        INPUT_FILE = name
        break

if not INPUT_FILE:
    print("📤 Upload pakistan_unique_osm_tags.csv:")
    uploaded = files.upload()
    INPUT_FILE = list(uploaded.keys())[0]
else:
    print(f"✅ Using: {INPUT_FILE}")

In [ ]:
import pandas as pd
import re
from collections import defaultdict

# Load the data
df = pd.read_csv(INPUT_FILE)
print(f" Loaded {len(df):,} tag pairs")
print(f"   Columns: {list(df.columns)}")
df.head()

In [ ]:
# ============================================
# CLEANING CONFIGURATION
# ============================================

# Values to completely remove (exact match, case-insensitive)
REMOVE_VALUES = {
    # Boolean/status values
    'yes', 'no', 'true', 'false', 'unknown', 'none', 'null', 'na', 'n/a',
    'undefined', 'default', 'other', 'maybe', 'probably',

    # Generic responses
    'ok', 'okay', 'done', 'pending', 'active', 'inactive',

    # Empty/placeholder
    '', ' ', '-', '_', '?', '???', 'fixme', 'todo', 'check',

    # Common generic values
    'main', 'general', 'standard', 'normal', 'common', 'typical',
    'both', 'all', 'any', 'some', 'various',

    # Directional (usually not semantic for corpus)
    'left', 'right', 'straight', 'forward', 'backward',
}

# Keys whose values should be kept entirely (even if numeric)
KEEP_VALUE_KEYS = {
    'name', 'name:en', 'name:ur', 'name:ar', 'name:pnb', 'name:pa', 'name:sd',
    'alt_name', 'alt_name:en', 'alt_name:ur',
    'old_name', 'official_name', 'short_name', 'loc_name',
    'description', 'note', 'comment',
    'brand', 'brand:en', 'brand:ur', 'operator',
    'ref', 'ref:ur',  # Route references like N-5, M-2
}

# Keys whose values are typically numeric/IDs (remove entirely)
NUMERIC_KEYS = {
    'building:levels', 'building:level', 'levels', 'level', 'layer',
    'lanes', 'lanes:forward', 'lanes:backward',
    'maxspeed', 'minspeed', 'maxheight', 'maxweight', 'maxwidth',
    'ele', 'elevation', 'height', 'width', 'length', 'depth',
    'voltage', 'frequency', 'gauge', 'cables', 'wires',
    'population', 'capacity', 'seats',
    'start_date', 'end_date', 'opening_date',
    'phone', 'fax', 'contact:phone',
    'addr:housenumber', 'addr:postcode', 'addr:unit',
    'passenger_lines', 'OBJECTID',
}

# Keys to skip entirely (metadata, not semantic)
SKIP_KEYS = {
    'source', 'source:imagery', 'source:geometry', 'source:ele', 'source2:map',
    'created_by', 'tiger:reviewed', 'tiger:source',
    'import', 'imported', 'is_in',
    'fixme', 'FIXME', 'note', 'notes', 'todo', 'TODO',
    'id', 'osm_id', 'ref:id',
    'wikidata', 'wikipedia', 'wikimedia_commons',
    'brand:wikidata', 'brand:wikipedia',
    'image', 'mapillary', 'survey:date',
}

# Minimum counts to keep (low-count tags are often errors)
MIN_COUNT = 5

# Minimum value length (single chars are usually codes)
MIN_VALUE_LENGTH = 2

print("✅ Cleaning rules configured")
print(f"   Values to remove: {len(REMOVE_VALUES)}")
print(f"   Keys with numeric values: {len(NUMERIC_KEYS)}")
print(f"   Keys to skip entirely: {len(SKIP_KEYS)}")

In [ ]:
def is_numeric(val):
    """Check if value is purely numeric (including decimals, negatives)."""
    if pd.isna(val):
        return True
    val_str = str(val).strip()
    # Match: 123, -123, 12.5, -12.5, 1,000, 123,456
    return bool(re.match(r'^-?[\d,]+\.?\d*$', val_str))

def is_url_or_reference(val):
    """Check if value is a URL, ID, or technical reference."""
    val_str = str(val).lower()
    patterns = [
        r'^https?://',  # URLs
        r'^www\.',
        r'^\d{4}-\d{2}-\d{2}',  # Dates
        r'^\+?\d[\d-]{5,}$',  # Phone numbers
        r'^[a-f0-9]{8,}$',  # Hex IDs
        r'^q\d+$',  # Wikidata IDs
    ]
    return any(re.match(p, val_str) for p in patterns)

def normalize_value(val):
    """Normalize a value for comparison."""
    if pd.isna(val):
        return ''
    return str(val).lower().strip()

def should_keep_tag(row):
    """Determine if a tag should be kept."""
    key = str(row.get('tag_key', '')).strip()
    value = str(row.get('tag_value', '')).strip()
    count = row.get('count', 0)
    value_lower = value.lower()

    # Check minimum count
    if count < MIN_COUNT:
        return False, 'low_count'

    # Skip entire keys
    if key in SKIP_KEYS or any(key.startswith(sk) for sk in ['source:', 'tiger:']):
        return False, 'skip_key'

    # Remove specific values
    if value_lower in REMOVE_VALUES:
        return False, 'remove_value'

    # Skip short values (unless in keep list)
    if len(value) < MIN_VALUE_LENGTH and key not in KEEP_VALUE_KEYS:
        return False, 'too_short'

    # Skip numeric values for numeric keys
    if key in NUMERIC_KEYS:
        return False, 'numeric_key'

    # Skip purely numeric values (unless in keep list or name fields)
    if is_numeric(value) and key not in KEEP_VALUE_KEYS:
        return False, 'numeric_value'

    # Skip URLs and references
    if is_url_or_reference(value) and 'name' not in key:
        return False, 'url_or_ref'

    return True, 'keep'

# Apply cleaning
print("🧹 Cleaning tags...")

removal_reasons = defaultdict(int)
keep_mask = []

for _, row in df.iterrows():
    keep, reason = should_keep_tag(row)
    keep_mask.append(keep)
    if not keep:
        removal_reasons[reason] += 1

df_clean = df[keep_mask].copy()

print(f"\n📊 Cleaning Results:")
print(f"   Original: {len(df):,} tags")
print(f"   Cleaned:  {len(df_clean):,} tags")
print(f"   Removed:  {len(df) - len(df_clean):,} tags")
print(f"\n📋 Removal breakdown:")
for reason, count in sorted(removal_reasons.items(), key=lambda x: -x[1]):
    print(f"   {reason}: {count:,}")

In [ ]:
def normalize_for_dedup(key, value):
    """Create normalized key for deduplication."""
    # Normalize key: lowercase, replace separators
    norm_key = key.lower().replace('_', ':').replace('-', ':')

    # Normalize value: lowercase, strip, normalize spaces
    norm_val = value.lower().strip()
    norm_val = re.sub(r'\s+', ' ', norm_val)  # Multiple spaces to single
    norm_val = re.sub(r'[_\-]', ' ', norm_val)  # Underscores/dashes to spaces

    return f"{norm_key}={norm_val}"

# Create normalized column
df_clean['normalized'] = df_clean.apply(
    lambda row: normalize_for_dedup(str(row['tag_key']), str(row['tag_value'])),
    axis=1
)

# Keep the one with highest count for each normalized form
df_dedup = df_clean.sort_values('count', ascending=False)
df_dedup = df_dedup.drop_duplicates(subset=['normalized'], keep='first')
df_dedup = df_dedup.drop(columns=['normalized'])

print(f"\n🔄 Deduplication:")
print(f"   Before: {len(df_clean):,} tags")
print(f"   After:  {len(df_dedup):,} tags")
print(f"   Duplicates removed: {len(df_clean) - len(df_dedup):,}")

In [ ]:
# Focus on semantically meaningful tags for corpus
# These are the key categories most useful for NLP/corpus work

CORPUS_KEY_PREFIXES = [
    # Primary semantic categories
    'amenity', 'shop', 'tourism', 'leisure', 'natural',
    'landuse', 'building', 'highway', 'railway', 'waterway',
    'historic', 'religious', 'place', 'office', 'craft',
    'healthcare', 'emergency', 'military', 'sport', 'cuisine',

    # Name fields (valuable for NLP)
    'name', 'alt_name', 'old_name', 'brand', 'operator',

    # Location context
    'addr:city', 'addr:suburb', 'addr:district', 'addr:province',
    'addr:street', 'addr:place', 'addr:region',

    # Surface/material (physical descriptions)
    'surface', 'material', 'roof:shape', 'roof:material',

    # Other semantic
    'religion', 'denomination', 'service', 'product',
]

def is_corpus_relevant(key):
    """Check if a key is relevant for corpus building."""
    key_lower = key.lower()
    return any(key_lower.startswith(prefix) or key_lower == prefix
               for prefix in CORPUS_KEY_PREFIXES)

# Create corpus-focused version
df_corpus = df_dedup[df_dedup['tag_key'].apply(is_corpus_relevant)].copy()

print(f"\n🎯 Corpus-Focused Tags:")
print(f"   Total cleaned: {len(df_dedup):,}")
print(f"   Corpus-relevant: {len(df_corpus):,}")

# Show distribution by key type
print("\n📊 Distribution by key type:")
key_dist = df_corpus['tag_key'].value_counts().head(20)
for key, cnt in key_dist.items():
    print(f"   {key}: {cnt:,}")

In [ ]:
# Show sample of cleaned corpus tags
print("\n📋 Sample Cleaned Tags:")
df_corpus.sort_values('count', ascending=False).head(30)

In [ ]:
# Save all versions

# 1. Full cleaned tags (everything that passed cleaning)
df_dedup.to_csv('pakistan_osm_tags_cleaned.csv', index=False)
print(f" Saved {len(df_dedup):,} cleaned tags to pakistan_osm_tags_cleaned.csv")

# 2. Corpus-focused tags (semantic subset)
df_corpus.to_csv('pakistan_osm_tags_corpus.csv', index=False)
print(f" Saved {len(df_corpus):,} corpus tags to pakistan_osm_tags_corpus.csv")

# 3. Just unique values (for vocabulary building)
unique_values = df_corpus['tag_value'].dropna().unique()
pd.DataFrame({'value': sorted(unique_values)}).to_csv('pakistan_osm_unique_values.csv', index=False)
print(f" Saved {len(unique_values):,} unique values to pakistan_osm_unique_values.csv")

In [ ]:
# Summary statistics
print("\n" + "="*60)
print(" FINAL SUMMARY")
print("="*60)
print(f"\nOriginal tags:        {len(df):>10,}")
print(f"After cleaning:       {len(df_clean):>10,}")
print(f"After deduplication:  {len(df_dedup):>10,}")
print(f"Corpus-relevant:      {len(df_corpus):>10,}")
print(f"Unique values:        {len(unique_values):>10,}")
print(f"\nReduction: {(1 - len(df_corpus)/len(df))*100:.1f}% from original")

In [ ]:
# Download all files
from google.colab import files

print("📥 Downloading files...")
files.download('pakistan_osm_tags_cleaned.csv')
files.download('pakistan_osm_tags_corpus.csv')
files.download('pakistan_osm_unique_values.csv')
print("✅ Done!")